# 🛡️ Autonomy Calibration — GRPO Training Notebook v2.0
**OpenEnv India Hackathon 2026**

## The Core Innovation
This notebook trains an agent on a **Partially Observable RL Benchmark** where:
- The same visible alert can have 2–3 completely different hidden root causes
- `INVESTIGATE` reveals the hidden truth at a small reward cost (-0.05)
- Blind correct guesses are penalized vs. informed correct decisions
- GRPO learns to use `INVESTIGATE` strategically, not randomly

## Baselines (Pre-established)
| Agent | email_triage | devops | financial | Strategy |
|-------|------------|--------|-----------|----------|
| **Blind** | 0.378 | 0.572 | 0.773 | Never investigates |
| **Smart** | 0.834 | 0.983 | 0.990 | Always investigates |
| **GRPO Target** | >0.85 | >0.99 | >0.99 | Selectively investigates |

**Setup:** `Runtime → Change runtime type → T4 GPU`

## Cell 1 — Install Dependencies

In [ ]:
!pip install --upgrade numpy trl transformers accelerate peft torchao bitsandbytes requests matplotlib
print('✅ Done. Please Restart Session if this is your first run!')

: 

## Cell 2 — Configuration

In [ ]:
import os, json, random, requests
import torch

# ── Live Environment API ──────────────────────────────────────────────────────
ENV_API = "https://joy0021-autonomy-calibration-benchmark.hf.space/api"
TASKS   = ["email_triage", "devops_incident", "financial_request"]

# Pre-established baselines (from running inference.py locally)
BLIND_BASELINE  = {"email_triage": 0.378, "devops_incident": 0.572, "financial_request": 0.773}
SMART_BASELINE  = {"email_triage": 0.834, "devops_incident": 0.983, "financial_request": 0.990}

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_ID        = "Qwen/Qwen2.5-0.5B-Instruct"  # ~1GB, fits in free T4
MAX_NEW_TOKENS  = 32
NUM_STEPS       = 120    # ~20 min on T4
LEARNING_RATE   = 5e-6
BATCH_SIZE      = 2
NUM_GENERATIONS = 4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f'Device  : {DEVICE}')
print(f'Model   : {MODEL_ID}')
print(f'Env API : {ENV_API}')

# Quick API sanity check
resp = requests.post(f'{ENV_API}/reset', json={'task': 'email_triage', 'seed': 0}, timeout=30)
obs = resp.json()
print(f'API status: {resp.status_code}')
print(f'Available actions: {obs["available_actions"]}')
print(f'Ambiguity hint present: {"confidence" in obs.get("prompt", "").lower()}')

## Cell 3 — Environment Helper Functions

In [ ]:
def env_reset(task: str, seed: int = None) -> dict:
    body = {'task': task}
    if seed is not None:
        body['seed'] = seed
    r = requests.post(f'{ENV_API}/reset', json=body, timeout=30)
    r.raise_for_status()
    return r.json()

def env_step(action_type: str) -> dict:
    r = requests.post(f'{ENV_API}/step', json={'type': action_type}, timeout=30)
    r.raise_for_status()
    return r.json()

def build_prompt(obs: dict) -> str:
    """Build LLM prompt that teaches the agent to consider investigating."""
    actions = obs.get('available_actions', [])
    situation = obs.get('prompt', '')
    investigate_note = ''
    if 'investigate' in actions:
        investigate_note = (
            "Note: You may use 'investigate' to reveal hidden information before deciding. "
            "This costs a small reward but can prevent costly wrong decisions.\n"
        )
    return (
        f"You are an AI agent making high-stakes decisions.\n"
        f"{investigate_note}"
        f"Situation:\n{situation}\n\n"
        f"Available actions: {actions}\n"
        f"Reply with ONLY one action name from the list above:"
    )

def extract_action(text: str, available: list) -> str:
    """Parse model output to find first valid action name."""
    text_lower = text.lower()
    # Prioritize 'investigate' if explicitly mentioned (key learning signal)
    if 'investigate' in text_lower and 'investigate' in available:
        return 'investigate'
    for a in available:
        if a in text_lower:
            return a
    return random.choice(available)

print('✅ Environment helpers ready')

## Cell 4 — Load Model + LoRA

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

print(f'Loading {MODEL_ID} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
)

lora_cfg = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05,
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
print('✅ Model ready')

## Cell 5 — Build Training Dataset

Each record is a step-0 prompt from the live environment.
Critically, ~70% of records are from HIGH-AMBIGUITY scenarios
where the agent MUST learn to investigate before acting.

In [ ]:
from datasets import Dataset

records = []
# Weight high-ambiguity scenarios more heavily for training
for task in TASKS:
    for seed in range(20):  # 20 seeds × 3 tasks = 60 records
        try:
            obs = env_reset(task, seed=seed)
            records.append({
                'prompt': build_prompt(obs),
                'task_id': task,
                'available_actions': json.dumps(obs.get('available_actions', [])),
                'seed': seed,
                'has_investigate': 'investigate' in obs.get('available_actions', []),
            })
        except Exception as e:
            print(f'Warning seed={seed} task={task}: {e}')

train_dataset = Dataset.from_list(records)
invest_count = sum(r['has_investigate'] for r in records)
print(f'✅ Dataset: {len(train_dataset)} total prompts')
print(f'   With INVESTIGATE available: {invest_count}/{len(records)} ({invest_count/len(records):.0%})')
print(train_dataset)

## Cell 6 — GRPO Reward Function

**Key design**: This reward function runs multi-step episodes.
When the model outputs 'investigate', the environment returns enriched
state without ending the episode. The agent then makes its real decision.
The TOTAL episode reward is what GRPO optimizes.

This directly trains the agent to learn:
- investigate first on ambiguous signals → higher total reward
- act directly on clear signals → avoids investigation cost

In [ ]:
def autonomy_reward_fn(completions, prompts=None, task_id=None,
                       available_actions=None, seed=None, **kwargs):
    """
    GRPO reward function for Autonomy Calibration.

    Runs a FULL multi-step episode for each completion:
    1. Extract action from model output
    2. If 'investigate' → step, get enriched obs, pick next action
    3. Return episode-level score

    This teaches the model that 'investigate' is a GATEWAY to higher rewards,
    not just a cost.
    """
    rewards = []
    n = len(completions)
    task_ids      = task_id if task_id else ['email_triage'] * n
    avail_list    = available_actions if available_actions else ['[]'] * n
    seed_list     = seed if seed else [None] * n

    for i, completion in enumerate(completions):
        if isinstance(completion, list):
            text = completion[0].get('content', '') if completion else ''
        else:
            text = str(completion)

        avail = json.loads(avail_list[i]) if isinstance(avail_list[i], str) else avail_list[i]
        ep_seed = seed_list[i] if i < len(seed_list) else None
        task = task_ids[i] if i < len(task_ids) else 'email_triage'

        try:
            # Fresh episode from same seed for reproducibility
            obs = env_reset(task, seed=ep_seed)
            current_avail = obs.get('available_actions', avail)
            action = extract_action(text, current_avail)

            episode_reward = 0.0
            max_steps_total = 8
            steps_taken = 0

            while steps_taken < max_steps_total:
                result = env_step(action)
                episode_reward += result.get('reward', {}).get('value', 0.01)
                done = result.get('done', False)
                info = result.get('info', {})

                # If we got episode_score from info, use that (more accurate)
                if done and info.get('episode_score') is not None:
                    episode_reward = info['episode_score']
                    break

                if done:
                    break

                # Get next observation and pick next action
                next_obs = result.get('observation', {})
                next_avail = next_obs.get('available_actions', [])
                if next_avail:
                    # For subsequent steps: use simple heuristic (GRPO only trains step 0)
                    next_action = next_avail[0]
                    for pref in ['classify_', 'diagnose_', 'fix_restart', 'verify_metrics',
                                 'approve_after', 'log_standard', 'close_completed', 'confirm']:
                        for a in next_avail:
                            if a.startswith(pref) or a == pref:
                                next_action = a
                                break
                    action = next_action
                steps_taken += 1

            rewards.append(float(min(0.99, max(0.01, episode_reward))))
        except Exception as e:
            rewards.append(0.01)   # API failure → minimum reward

    return rewards

print('✅ Reward function defined')
print('   Each completion → full episode → calibration-weighted reward in [0.01, 0.99]')

## Cell 7 — Configure & Run GRPO Training
⏱️ Expected time on T4: ~20 minutes for 120 steps.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

# 1. ⚙️ Hyperparameters
NUM_STEPS = 120
BATCH_SIZE = 1
NUM_GENERATIONS = 8
LEARNING_RATE = 5e-6
MAX_COMPLETION_LENGTH = 128

# 2. 🛠️ Training Configuration
config = GRPOConfig(
    output_dir                  = './grpo_checkpoints',
    num_generations             = NUM_GENERATIONS,
    generation_batch_size       = NUM_GENERATIONS,
    max_steps                   = NUM_STEPS,
    per_device_train_batch_size  = BATCH_SIZE,
    learning_rate               = LEARNING_RATE,
    logging_steps               = 5,
    save_steps                  = 40,
    gradient_accumulation_steps = 2,
    report_to                   = 'none',
)

# 3. 🚀 Initialize Trainer
trainer = GRPOTrainer(
    model                 = model,
    processing_class      = tokenizer,
    reward_funcs          = autonomy_reward_fn,
    args                  = config,
    train_dataset         = train_dataset,
    max_prompt_length     = 512,
    max_completion_length = MAX_COMPLETION_LENGTH,
)

print(f'🚀 Starting GRPO training ({NUM_STEPS} steps)...')
trainer.train()
print('\n✅ Training complete!')

## Cell 8 — Plot Training Curves

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

log = trainer.state.log_history
steps   = [e['step']   for e in log if 'loss' in e]
losses  = [e['loss']   for e in log if 'loss' in e]
r_steps = [e['step']   for e in log if 'reward' in e]
rewards = [e['reward'] for e in log if 'reward' in e]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Autonomy Calibration — GRPO Training Evidence', fontsize=14, fontweight='bold')

# Loss curve
axes[0].plot(steps, losses, color='#E74C3C', linewidth=2)
axes[0].set_title('Policy Loss')
axes[0].set_xlabel('Training Step')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

# Reward curve with baselines
blind_avg  = sum(BLIND_BASELINE.values()) / len(BLIND_BASELINE)
smart_avg  = sum(SMART_BASELINE.values()) / len(SMART_BASELINE)

if rewards:
    w = 5
    smooth = np.convolve(rewards, np.ones(w)/w, mode='valid')
    axes[1].plot(r_steps[w-1:], smooth, color='#27AE60', linewidth=2, label='GRPO Agent')
axes[1].axhline(y=blind_avg,  color='#E74C3C',  linestyle='--', linewidth=2, label=f'Blind Baseline ({blind_avg:.3f})')
axes[1].axhline(y=smart_avg,  color='#3498DB',  linestyle='-.',  linewidth=2, label=f'Smart Baseline ({smart_avg:.3f})')
axes[1].set_title('Episode Reward')
axes[1].set_xlabel('Training Step')
axes[1].set_ylabel('Reward (0.01–0.99)')
axes[1].set_ylim(0.0, 1.05)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.close()
print('✅ Saved training_curves.png')

## Cell 9 — Evaluate Trained Agent vs Baselines

In [ ]:
def eval_agent(task, n_episodes=10):
    scores = []
    investigated_count = 0
    for ep in range(n_episodes):
        obs  = env_reset(task, seed=ep)
        ep_score = 0.0
        investigated_this_ep = False
        steps = 0

        while not obs.get('done', False) and steps < 10:
            prompt = build_prompt(obs)
            inputs = tokenizer(prompt, return_tensors='pt',
                               truncation=True, max_length=512).to(DEVICE)
            with torch.no_grad():
                out = model.generate(
                    **inputs, max_new_tokens=MAX_NEW_TOKENS,
                    temperature=0.3, do_sample=True,
                    pad_token_id=tokenizer.eos_token_id
                )
            text    = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
            avail   = obs.get('available_actions', [])
            action  = extract_action(text, avail)

            if action == 'investigate':
                investigated_this_ep = True

            result  = env_step(action)
            info    = result.get('info', {})
            if 'episode_score' in info and info['episode_score'] is not None:
                ep_score = info['episode_score']
            obs  = result.get('observation', {})
            if result.get('done', False):
                break
            steps += 1

        scores.append(ep_score)
        if investigated_this_ep:
            investigated_count += 1

    return {
        'avg': sum(scores)/len(scores) if scores else 0.0,
        'investigate_rate': investigated_count/n_episodes,
        'scores': scores,
    }

print('Evaluating trained agent (10 episodes/task)...')
results = {}
for task in TASKS:
    r = eval_agent(task, n_episodes=10)
    results[task] = r
    print(f'  {task}: score={r["avg"]:.3f} | investigate_rate={r["investigate_rate"]:.0%}')
    print(f'    blind_baseline={BLIND_BASELINE[task]:.3f} | smart_baseline={SMART_BASELINE[task]:.3f}')
    delta = r['avg'] - BLIND_BASELINE[task]
    print(f'    Improvement over blind: {delta:+.3f}')

## Cell 10 — Final Comparison Chart

In [ ]:
labels  = [t.replace('_', '\n') for t in TASKS]
blind   = [BLIND_BASELINE[t] for t in TASKS]
smart   = [SMART_BASELINE[t] for t in TASKS]
trained = [results[t]['avg'] for t in TASKS]

x = np.arange(len(labels))
w = 0.25

fig, ax = plt.subplots(figsize=(11, 6))
b1 = ax.bar(x - w,   blind,   w, label='Blind (never investigates)',  color='#E74C3C', alpha=0.85)
b2 = ax.bar(x,       smart,   w, label='Smart (always investigates)', color='#3498DB', alpha=0.85)
b3 = ax.bar(x + w,   trained, w, label='GRPO Trained (selective)',    color='#27AE60', alpha=0.85)

for bar_group in [b1, b2, b3]:
    for bar in bar_group:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_title('Autonomy Calibration Benchmark\nBaseline vs GRPO Trained Agent',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Average Episode Reward (0.01–0.99)', fontsize=11)
ax.set_ylim(0, 1.15)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Add investigate rate annotations
for i, task in enumerate(TASKS):
    rate = results[task]['investigate_rate']
    ax.text(i + w, 0.05, f'Inv: {rate:.0%}', ha='center', fontsize=8, color='white', fontweight='bold')

plt.tight_layout()
plt.savefig('baseline_vs_trained.png', dpi=150, bbox_inches='tight')
plt.close()
print('✅ Saved baseline_vs_trained.png')

# Print final validation
print('\n=== FINAL VALIDATION CHECKLIST ===')
for task in TASKS:
    r = results[task]
    beats_blind = r['avg'] > BLIND_BASELINE[task]
    uses_invest = r['investigate_rate'] > 0.30
    print(f'{task}:')
    print(f'  [{'✅' if beats_blind else '❌'}] Beats blind baseline ({r["avg"]:.3f} > {BLIND_BASELINE[task]:.3f})')
    print(f'  [{'✅' if uses_invest else '❌'}] Uses investigate strategically ({r["investigate_rate"]:.0%} rate)')

## Cell 11 — Download Plots

In [ ]:
from google.colab import files
for f in ['training_curves.png', 'baseline_vs_trained.png']:
    files.download(f)
    print(f'⬇️  Downloaded {f}')

## Cell 12 — (Optional) Push Model to HF Hub

In [ ]:
from huggingface_hub import login

HF_TOKEN = 'your_new_hf_write_token_here'
HF_REPO  = 'JOY0021/autonomy-grpo-agent-v2'

if HF_TOKEN != 'your_new_hf_write_token_here':
    login(token=HF_TOKEN)
    trainer.model.push_to_hub(HF_REPO)
    tokenizer.push_to_hub(HF_REPO)
    print(f'✅ Model → https://huggingface.co/{HF_REPO}')
else:
    print('⚠️  Add HF token to push model. Skipping.')